# 0 - Load the data

In [8]:
!pip install pypdf

In [60]:
import numpy as np

In [9]:
from pypdf import PdfReader
from os import walk

# Getting the files
path = './resources/'
files = []
for (dirpath, dirnames, filenames) in walk(path):
    files.extend(filenames)

print(files)

# Parsing PDF
files_content = {}
for f in files:
    print(f"Processing {path + f}...")
    pdf_reader = PdfReader(path + f)
    content = ""
    for page in pdf_reader.pages:
        content += page.extract_text(0) + "\n"
    files_content[f] = content

print(files_content.values())

['2510.23590v1.pdf', '2510.23595v1.pdf', '2510.23596v1.pdf', '2510.23601v1.pdf', '2510.23603v1.pdf']
Processing ./resources/2510.23590v1.pdf...
Processing ./resources/2510.23595v1.pdf...
Processing ./resources/2510.23596v1.pdf...
Processing ./resources/2510.23601v1.pdf...
Processing ./resources/2510.23603v1.pdf...
dict_values(['Lightweight Robust Direct Preference Optimization\nCheol Woo Kim∗ Shresth Verma* Mauricio Tec* Milind Tambe\nSchool of Engineering and Applied Sciences, Harvard University, Boston, MA 02134\n{cwkim, sverma, mauriciogtec, tambe}@g.harvard.edu\nAbstract\nDirect Preference Optimization (DPO) has become a popular method for fine-\ntuning large language models (LLMs) due to its stability and simplicity. However,\nit is also known to be sensitive to noise in the data and prone to overfitting. Recent\nworks have proposed using distributionally robust optimization (DRO) to address\npotential noise and distributional shift in the data. However, these methods often\nsuffe

In [61]:
class Chunk:
    def __init__(self, text, file):
        self.text = text
        self.file = file
        self.embed = None


# 1 - Chunking

In [62]:
chunks = []
chunk_size = 1024
for (file, content) in files_content.items():
    for i in range(0, len(content), chunk_size):
        chunks.append(Chunk(content[i:i + chunk_size], file))

# 2 - Embedding of documents

In [ ]:
# Pull an embedding model with ollama
!ollama pull mxbai-embed-large

In [63]:
from langchain_ollama import OllamaEmbeddings

In [56]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large")
chunks_text = list(map(lambda c: c.text, chunks))
embeddings = embedding_model.embed_documents(chunks_text)

In [64]:
for i in range(0, len(chunks)):
    chunks[i].embed = np.array(embeddings[i])

# 3 - Initializing Ollama for chat

In [65]:
from langchain_ollama import ChatOllama

chat_model = ChatOllama(model="qwen3:4b")

# 4 - Embed the prompt, fetch the relevant documents (retrieval) and add it to the context (augmented generation)

In [66]:
prompt = "What is DPO-PRO ?"

In [67]:
prompt_embedding = np.array(embedding_model.embed_query(prompt))

In [68]:
chunks.sort(key=lambda c: np.dot(prompt_embedding, c.embed) / (np.linalg.norm(prompt_embedding) * np.linalg.norm(c.embed)), reverse=True)

In [73]:
extended_prompt = "<context>\n"
for i in range(0, 5):
    extended_prompt += f"<document #{i}>{chunks[i].text}</document #{i}>\n"
extended_prompt += "</context>\n"
extended_prompt += prompt

print(extended_prompt)

<context>
<document #0> DPO-PRO outperforms vanilla DPO and DrDPO.
8
Table 2: Win rate (%) based on LLM-as-a-judge evaluation for the reward function design task (↑).
Noise Setting DPO DrDPO DPO-PRO
No noise (α= 0) 55.0 40.0 35.3
Low noise (α= 0.3) 45.0 54.8 63.9
Table 3: Examples of reward functions produced by vanilla-DPO and DPO-PRO under noisy
training conditions. s indicates binary engagement status, and other input features (e.g., demographic
attributes or call timing) are also encoded as binary variables. DPO-PRO produces more conservative
interpretations of preferences.
Task 1:Prefer both young and elderly beneficiaries
Vanilla-DPOs + 3 * (youngest_age or second_youngest_age or
oldest_age) +
2 * (lowest_education or second_lowest_education
or third_lowest_education) +
(lowest_income or second_lowest_income or
third_lowest_income)
DPO-PROs + 3 * (youngest_age or second_youngest_age or
oldest_age) +
2 * (lowest_education or second_lowest_education
or third_lowest_education)
Task 

In [74]:
chat_model.invoke(extended_prompt)

AIMessage(content="# What is DPO-PRO?\n\nDPO-PRO (DPO with Preference Robustness) is a **distributionally robust enhancement of the DPO (Direct Preference Optimization)** method that specifically targets uncertainties in the preference distribution. \n\nBased on the provided context, DPO-PRO is designed to:\n\n1. Incorporate uncertainty in the preference distribution using an efficient DRO (Distributionally Robust Optimization) formulation\n2. Focus solely on robustness to the preference distribution (rather than the full data-generating process)\n3. Avoid excessive conservatism that other DRO-based approaches might introduce\n4. Have negligible computational overhead while improving model reliability under imperfect preference annotation\n\nThe key innovation of DPO-PRO is that it creates a robust learning framework that remains reliable even when there's noise in the preference annotations (measured by parameter α in the context). This is particularly important in real-world applicat